# 03 — DDI Safety Evaluation

Evaluate drug-drug interaction rates for drug recommendation models.
DDI rate measures what fraction of recommended drug pairs have known interactions — lower is safer.

Compare SafeDrug (DDI-constrained) vs GAMENet (accuracy-focused).

In [ ]:
from pyhealth_enterprise.datasets.synthetic import SyntheticEHRDataset
from pyhealth.tasks import drug_recommendation_mimic3_fn
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.models import SafeDrug, GAMENet
from pyhealth.trainer import Trainer
from pyhealth.metrics import ddi_rate_score, multilabel_metrics_fn
import pandas as pd

ds = SyntheticEHRDataset(); ds.load()
task_dataset = ds.dataset.set_task(drug_recommendation_mimic3_fn)
train, val, test = split_by_patient(task_dataset, [0.8, 0.1, 0.1])
train_loader = get_dataloader(train, batch_size=32, shuffle=True)
val_loader   = get_dataloader(val,   batch_size=32, shuffle=False)
test_loader  = get_dataloader(test,  batch_size=32, shuffle=False)

In [ ]:
all_results = {}
for ModelCls, name in [(SafeDrug, 'SafeDrug'), (GAMENet, 'GAMENet')]:
    kwargs = {}
    if ModelCls == SafeDrug:
        kwargs = {'ddi_adj': task_dataset.ddi_adj, 'ddi_mask_H': task_dataset.ddi_mask_H}
    model = ModelCls(dataset=task_dataset, feature_keys=['conditions','drugs'],
                     label_key='drugs', mode='multilabel', **kwargs)
    trainer = Trainer(model=model, metrics=['jaccard', 'prauc', 'f1'])
    trainer.train(train_dataloader=train_loader, val_dataloader=val_loader,
                  epochs=30, monitor='jaccard')
    r = trainer.evaluate(test_loader)
    r['ddi_rate'] = ddi_rate_score(r['y_prob'], task_dataset.ddi_adj)
    all_results[name] = {k: v for k, v in r.items() if k not in ['y_prob', 'y_true']}

df = pd.DataFrame(all_results).T.round(4)
print(df)
print('\nLower DDI rate = safer recommendations')